# FinWolf OCR — Training on Kaggle (T4 GPU)
**Steps:** Install deps → Generate training data → Fine-tune Qwen2.5-14B → Download adapter

In [ ]:
# Step 1 — Install dependencies
!pip install -q transformers peft datasets accelerate bitsandbytes trl sentencepiece

In [ ]:
# Step 2 — Load the text files uploaded directly to Kaggle dataset
import os, json, random
from pathlib import Path

# Try common locations Kaggle puts uploaded files
TEXT_DIR = None
for candidate in [
    Path('/kaggle/input/finwolf-raw-text'),
    Path('/kaggle/input/finwolf-raw-text/raw_text'),
    Path('/kaggle/input/finwolf-raw-text/data/raw_text'),
]:
    files = sorted(candidate.glob('*.txt')) if candidate.exists() else []
    if files:
        TEXT_DIR = candidate
        break

if TEXT_DIR is None:
    # Last resort: search all input folders
    for root, dirs, files in os.walk('/kaggle/input'):
        txts = [f for f in files if f.endswith('.txt')]
        if txts:
            TEXT_DIR = Path(root)
            break

txt_files = sorted(TEXT_DIR.glob('*.txt'))
print(f'Found {len(txt_files)} text files in {TEXT_DIR}')

In [ ]:
# Step 3 — Generate training data using Qwen2.5-7B (fast inference on T4)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

TEACHER = 'Qwen/Qwen2.5-7B-Instruct'

print('Loading teacher model (Qwen2.5-7B)...')
teacher_tok = AutoTokenizer.from_pretrained(TEACHER)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER,
    torch_dtype=torch.float16,
    device_map='auto'
)
teacher_model.eval()
print('Teacher ready')

In [ ]:
SYSTEM_PROMPT = """Du bist ein Spezialist fuer Schweizer Versicherungs- und Finanzdokumente.
Extrahiere die angeforderten Felder und antworte ausschliesslich mit einem JSON-Objekt.

Format:
{"document_type": "string", "insurer": "string or null", "document_language": "DE|FR|IT|EN",
 "extracted_fields": [{"field": "Name", "value": "Wert", "confidence": "high|medium|low", "page": 1}],
 "not_found": ["Feld1"], "summary": "Kurze Zusammenfassung"}"""

# 3 queries instead of 6 — covers all key fields, cuts time in half
QUERIES = [
    ('personal',  'Name, Vorname, Geburtsdatum, Adresse, AHV-Nummer, Versicherten-Nummer'),
    ('premiums',  'Monatspraemie, Jahrespraemie, Franchise, Selbstbehalt, Police Nr., Versicherungsgesellschaft, Versicherungsbeginn'),
    ('all',       'Alle relevanten Felder: Police/Vertragsnummer, Versicherungsnehmer, Geburtsdatum, Versicherungsgesellschaft, alle Praemienbetraege, Franchise, Selbstbehalt, Vertragsbeginn, Deckungsumfang, Spitalabteilung, Kennzeichen'),
]

def generate_response(doc_text, query):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Dokument:\n{doc_text[:3000]}\n\nExtrahiere: {query}'}
    ]
    prompt = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = teacher_tok(prompt, return_tensors='pt').to(teacher_model.device)
    with torch.no_grad():
        out = teacher_model.generate(**inputs, max_new_tokens=800, do_sample=False,
                                      pad_token_id=teacher_tok.eos_token_id)
    return teacher_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try: return json.loads(raw)
    except: return None

print('Query functions ready — 3 queries per document')

In [ ]:
# Generate training data — batched inference (4 docs at once = 3-4x faster)
from tqdm import tqdm

BATCH_SIZE = 4
teacher_tok.padding_side = 'left'
if teacher_tok.pad_token is None:
    teacher_tok.pad_token = teacher_tok.eos_token

def generate_batch(batch_texts, query):
    messages_list = [
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': f'Dokument:\n{text[:3000]}\n\nExtrahiere: {query}'}]
        for text in batch_texts
    ]
    prompts = [teacher_tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    inputs = teacher_tok(
        prompts, return_tensors='pt', padding=True,
        truncation=True, max_length=2048
    ).to(teacher_model.device)
    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs, max_new_tokens=800, do_sample=False,
            pad_token_id=teacher_tok.eos_token_id
        )
    input_len = inputs['input_ids'].shape[1]
    return [
        teacher_tok.decode(out[input_len:], skip_special_tokens=True).strip()
        for out in outputs
    ]

examples = []
failed = 0

# Split files into batches
batches = [txt_files[i:i+BATCH_SIZE] for i in range(0, len(txt_files), BATCH_SIZE)]

for batch_paths in tqdm(batches, desc='Batches'):
    batch_texts = [p.read_text(encoding='utf-8', errors='ignore') for p in batch_paths]
    for query_name, query_fields in QUERIES:
        raws = generate_batch(batch_texts, query_fields)
        for txt_path, raw in zip(batch_paths, raws):
            doc_text = txt_path.read_text(encoding='utf-8', errors='ignore')
            parsed = parse_json(raw)
            if parsed is None:
                failed += 1
                continue
            examples.append({
                'messages': [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f'Dokument:\n{doc_text[:3000]}\n\nExtrahiere: {query_fields}'},
                    {'role': 'assistant', 'content': json.dumps(parsed, ensure_ascii=False)},
                ]
            })

random.shuffle(examples)
split = int(len(examples) * 0.9)
train_data, valid_data = examples[:split], examples[split:]

Path('/kaggle/working/finwolf_train.jsonl').write_text(
    '\n'.join(json.dumps(e, ensure_ascii=False) for e in train_data))
Path('/kaggle/working/finwolf_valid.jsonl').write_text(
    '\n'.join(json.dumps(e, ensure_ascii=False) for e in valid_data))

print(f'Done: {len(train_data)} train + {len(valid_data)} valid examples ({failed} failed)')

In [ ]:
# Step 4 — Free teacher model memory before fine-tuning
import gc
del teacher_model
gc.collect()
torch.cuda.empty_cache()
print('GPU memory freed')

In [ ]:
import gc, torch, json
from pathlib import Path
from datasets import load_dataset
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

train_file = Path('/kaggle/working/finwolf_train.jsonl')
valid_file = Path('/kaggle/working/finwolf_valid.jsonl')
print(f'Train: {sum(1 for _ in open(train_file))} examples')
print(f'Valid: {sum(1 for _ in open(valid_file))} examples')

STUDENT = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
student_tok = AutoTokenizer.from_pretrained(STUDENT)
if student_tok.pad_token is None:
    student_tok.pad_token = student_tok.eos_token

print('Loading Qwen2.5-7B...')
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT, quantization_config=bnb_config,
    device_map='cuda:0', trust_remote_code=True,
)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05, target_modules=['q_proj','v_proj','k_proj','o_proj'], bias='none',
)
student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

dataset = load_dataset('json', data_files={
    'train': str(train_file), 'validation': str(valid_file)
})
def format_chat(example):
    return {'text': student_tok.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(format_chat)

training_args = SFTConfig(
    output_dir='/kaggle/working/finwolf-7b-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    evaluation_strategy='steps',
    eval_steps=50,
    max_seq_length=1024,
    report_to='none',
)
trainer = SFTTrainer(
    model=student_model, tokenizer=student_tok, args=training_args,
    train_dataset=dataset['train'], eval_dataset=dataset['validation'],
)
print('Starting fine-tuning...')
trainer.train()
print('Done!')

import shutil
student_model.save_pretrained('/kaggle/working/finwolf-7b-adapter')
student_tok.save_pretrained('/kaggle/working/finwolf-7b-adapter')
shutil.make_archive('/kaggle/working/finwolf-7b-adapter', 'zip', '/kaggle/working/finwolf-7b-adapter')
size = Path('/kaggle/working/finwolf-7b-adapter.zip').stat().st_size / 1e6
print(f'Adapter saved — {size:.0f} MB. Download from Output tab.')

In [ ]:
# Step 6 — Fine-tune with SFTTrainer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset('json', data_files={
    'train': '/kaggle/working/finwolf_train.jsonl',
    'validation': '/kaggle/working/finwolf_valid.jsonl'
})

def format_chat(example):
    return {'text': student_tok.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_chat)

training_args = SFTConfig(
    output_dir='/kaggle/working/finwolf-14b-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    evaluation_strategy='steps',
    eval_steps=50,
    max_seq_length=1024,
    report_to='none',
)

trainer = SFTTrainer(
    model=student_model,
    tokenizer=student_tok,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
)

print('Starting fine-tuning...')
trainer.train()
print('Fine-tuning complete!')

In [ ]:
# Step 7 — Save adapter and zip for download
import shutil

student_model.save_pretrained('/kaggle/working/finwolf-14b-adapter')
student_tok.save_pretrained('/kaggle/working/finwolf-14b-adapter')

shutil.make_archive('/kaggle/working/finwolf-14b-adapter', 'zip', '/kaggle/working/finwolf-14b-adapter')
print('Saved! Download finwolf-14b-adapter.zip from the Output tab')

import os
size = os.path.getsize('/kaggle/working/finwolf-14b-adapter.zip') / 1e6
print(f'Adapter size: {size:.0f} MB')